# CREATING FULL TABLE

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast

In [2]:
mol_type = pd.read_csv("../tables/mol_type_parsed.csv")
gff_parsed = pd.read_csv("../tables/gff_parsed.csv")
assembly = pd.read_csv("../tables/assembly_data_report.csv")
cluster = pd.read_csv("../tables/clusterRes_cluster.tsv", sep='\t', names=["cluster", "ID"])
accession_record = pd.read_csv("../tables/accession_record_table.csv")

/var/folders/vc/z6j_dxbj4wl_jhtyr4jzsbw40000gn/T/ipykernel_93299/3326860764.py:1: DtypeWarning: Columns (0: region_strand) have mixed types. Specify dtype option on import or set low_memory=False.
  mol_type = pd.read_csv("../tables/mol_type_parsed.csv")


In [61]:
gff_parsed

,Unnamed: 0,record_id,type,start,end,strand,qualifiers,search_term,source_file
0,0,MG982796.1,CDS,25,594,1,"{'ID': ['cds-AVT55874.1'], 'Parent': ['gene-PA...",slip,../viral_data_all/ncbi_dataset/data_subset/GCA...
1,1,MG982796.1,CDS,596,724,1,"{'ID': ['cds-AVT55874.1'], 'Parent': ['gene-PA...",slip,../viral_data_all/ncbi_dataset/data_subset/GCA...
2,2,MT090511.1,CDS,1,570,1,"{'ID': ['cds-QIC52522.1'], 'Parent': ['gene-PA...",slip,../viral_data_all/ncbi_dataset/data_subset/GCA...
3,3,MT090511.1,CDS,572,760,1,"{'ID': ['cds-QIC52522.1'], 'Parent': ['gene-PA...",slip,../viral_data_all/ncbi_dataset/data_subset/GCA...
4,4,MK624178.1,CDS,13,582,1,"{'ID': ['cds-QBL91539.1'], 'Parent': ['gene-PA...",slip,../viral_data_all/ncbi_dataset/data_subset/GCA...
...,...,...,...,...,...,...,...,...,...
251141,251231,PQ011362.1,CDS,572,760,1,"{'ID': ['cds-XCO67773.1'], 'Parent': ['gene-PA...",slip,../viral_data_all/ncbi_dataset/data_subset/GCA...
251142,251232,AJ302647.1,CDS,843,2151,1,"{'ID': ['cds-CAC38421.2'], 'Parent': ['gene-ga...",slip,../viral_data_all/ncbi_dataset/data_subset/GCA...
251143,251233,AJ302647.1,CDS,2150,5180,1,"{'ID': ['cds-CAC38421.2'], 'Parent': ['gene-ga...",slip,../viral_data_all/ncbi_dataset/data_subset/GCA...
251144,251234,AJ302646.1,CDS,843,2151,1,"{'ID': ['cds-CAC38430.2'], 'Parent': ['gene-ga...",slip,../viral_data_all/ncbi_dataset/data_subset/GCA...


# GFF Cleaning


In [62]:
gff_parsed['qualifiers'] = gff_parsed['qualifiers'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
gff_parsed['exception'] = gff_parsed['qualifiers'].apply(lambda x: x.get('exception', [None])[0])

In [63]:
gff_parsed = gff_parsed[gff_parsed['exception'].str.contains('ibosom', case=False, na=False)]

In [64]:
gff_parsed[gff_parsed['record_id'] == 'MW159331.1']

,Unnamed: 0,record_id,type,start,end,strand,qualifiers,search_term,source_file,exception
5305,5306,MW159331.1,CDS,25,594,1,"{'ID': ['cds-QOQ30264.1'], 'Parent': ['gene-PA...",slip,../viral_data_all/ncbi_dataset/data_subset/GCA...,ribosomal slippage
5306,5307,MW159331.1,CDS,596,784,1,"{'ID': ['cds-QOQ30264.1'], 'Parent': ['gene-PA...",slip,../viral_data_all/ncbi_dataset/data_subset/GCA...,ribosomal slippage


In [65]:
gff_parsed.record_id.value_counts()

record_id
OP785764.1    19
OP785763.1    19
OM112288.1    19
OP785766.1    19
MZ366331.1    19
              ..
AY008716.1     1
AY616033.2     1
PQ443910.1     1
AB753015.2     1
PQ510499.1     1
Name: count, Length: 125031, dtype: int64

In [67]:
gff_parsed.record_id.value_counts().value_counts().sort_index()

count
1         24
2     124879
3         14
4         49
5          1
6          3
8          3
12         1
13        36
14         9
19        12
Name: count, dtype: int64

In [ ]:
#gff_parsed[gff_parsed["record_id"] == 'OP785764.1']
#gff_parsed.qualifiers.iloc[0]

In [68]:
counts = gff_parsed.record_id.value_counts()
gff_clean = gff_parsed[gff_parsed.record_id.isin(counts[counts == 2].index)]
gff_clean = gff_clean[gff_clean.type == 'CDS']


In [ ]:
#gff_clean[gff_clean.record_id == 'AY357582.2']

In [71]:
gff_collapsed = (
    gff_clean.groupby(['record_id', 'strand', 'source_file', 'search_term'])
    .agg(
        type       = ('type',  'first'),
        section1   = ('start', list),
        section2   = ('end',   list),
        n_segments = ('start', 'count'),
        qualifiers = ('qualifiers', 'first'),
        exception  = ('exception', 'first'),
    )
    .reset_index()
)

gff_collapsed['start1'] = gff_collapsed['section1'].apply(lambda x: x[0] if len(x) > 0 else None)
gff_collapsed['end1']   = gff_collapsed['section2'].apply(lambda x: x[0] if len(x) > 0 else None)
gff_collapsed['start2'] = gff_collapsed['section1'].apply(lambda x: x[1] if len(x) > 1 else None)
gff_collapsed['end2']   = gff_collapsed['section2'].apply(lambda x: x[1] if len(x) > 1 else None)

gff_collapsed = gff_collapsed.drop(columns=['section1', 'section2'])

print(f"Before: {len(gff_clean)} rows → After: {len(gff_collapsed)} rows")

Before: 249756 rows → After: 124878 rows


In [72]:
mask = gff_collapsed['strand'] == -1

gff_collapsed.loc[mask, ['start1', 'end1', 'start2', 'end2']] = (
    gff_collapsed.loc[mask, ['start2', 'end2', 'start1', 'end1']].values
)

In [75]:
gff_collapsed['slippage_direction'] = gff_collapsed.apply(lambda row: row['start2'] - row['end1'] - 1, axis=1)

In [20]:
gff_collapsed['slippage_site'] = gff_collapsed['end1']

In [22]:
gff_collapsed.to_csv("../tables/CLEAN_gff2.csv")

In [3]:
gff_collapsed = pd.read_csv('../tables/CLEAN_gff2.csv')

In [21]:
gff_collapsed

,Unnamed: 0,record_id,strand,source_file,search_term,type,n_segments,qualifiers,exception,start1,end1,start2,end2,slippage_direction,slippage_site
0,0,AB033998.1,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,slip,CDS,2,"{'ID': ['cds-BAA92848.1'], 'Dbxref': ['NCBI_GP...",ribosomal slippage,14,3025,3025,4551,-1,3025
1,1,AB523788.1,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,slip,CDS,2,"{'ID': ['cds-BAI77525.1'], 'Dbxref': ['NCBI_GP...",ribosomal slippage,74,6034,6036,7553,1,6034
2,2,AB594828.1,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,slip,CDS,2,"{'ID': ['cds-BAJ72195.1'], 'Parent': ['gene-OR...",ribosomal slippage,176,1621,1621,3432,-1,1621
3,3,AF022214.2,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,slip,CDS,2,"{'ID': ['cds-AAC18466.2'], 'Parent': ['gene-PB...",ribosomal slippage,15695,16075,16075,16518,-1,16075
4,4,AF033808.1,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,slip,CDS,2,"{'ID': ['cds-AAC82561.1'], 'Dbxref': ['NCBI_GP...",ribosomal slippage,380,2482,2482,5190,-1,2482
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124873,124873,PZ005496.1,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,slip,CDS,2,"{'ID': ['cds-YDB37122.1'], 'Parent': ['gene-PA...",ribosomal slippage,13,582,584,772,1,582
124874,124874,PZ005504.1,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,slip,CDS,2,"{'ID': ['cds-YDB37134.1'], 'Parent': ['gene-PA...",ribosomal slippage,13,582,584,772,1,582
124875,124875,PZ005542.1,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,slip,CDS,2,"{'ID': ['cds-YDB37190.1'], 'Parent': ['gene-PA...",ribosomal slippage,13,582,584,772,1,582
124876,124876,U51190.1,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,slip,CDS,2,"{'ID': ['cds-AAC97542.1'], 'Dbxref': ['NCBI_GP...",ribosomal slippage,160,1448,1448,3146,-1,1448


In [ ]:
cluster

,cluster,ID
0,MW334171.1,MW334171.1
1,MW334171.1,OP023516.1
2,MW334171.1,PQ755305.1
3,MW334171.1,PQ756818.1
4,MW334171.1,PQ756310.1
...,...,...
1326720,KP939613.1,KP939837.1
1326721,KP939613.1,KP940036.1
1326722,KP939613.1,KP940044.1
1326723,OP890528.1,OP890528.1


In [21]:
accession_record

,accession,record_id
0,GCA_000320725.1,AFYC01000010.1
1,GCA_000320725.1,AFYC01000009.1
2,GCA_000320725.1,AFYC01000008.1
3,GCA_000320725.1,AFYC01000007.1
4,GCA_000320725.1,AFYC01000006.1
...,...,...
1443825,GCF_004787635.1,NC_075420.1
1443826,GCF_023124225.1,NC_076668.1
1443827,GCF_029888005.1,NC_078996.1
1443828,GCF_029888005.1,NC_078994.1


In [23]:
full = cluster.merge(accession_record, how = 'left', left_on = 'ID', right_on = 'record_id').drop(columns = 'ID')

In [6]:
full

,cluster,accession,record_id
0,MW334171.1,GCA_038804155.1,MW334171.1
1,MW334171.1,GCA_039240725.1,OP023516.1
2,MW334171.1,GCA_046381355.1,PQ755305.1
3,MW334171.1,GCA_046380515.1,PQ756818.1
4,MW334171.1,GCA_046378115.1,PQ756310.1
...,...,...,...
1326981,KP939613.1,GCA_003084715.1,KP939837.1
1326982,KP939613.1,GCA_003085195.1,KP940036.1
1326983,KP939613.1,GCA_003085355.1,KP940044.1
1326984,OP890528.1,GCA_027132475.1,OP890528.1


In [7]:
gff_collapsed

,Unnamed: 0,record_id,strand,source_file,search_term,type,n_segments,qualifiers,exception,start1,end1,start2,end2,slippage_direction
0,0,AB033998.1,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,slip,CDS,2,"{'ID': ['cds-BAA92848.1'], 'Dbxref': ['NCBI_GP...",ribosomal slippage,14,3025,3025,4551,-1
1,1,AB523788.1,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,slip,CDS,2,"{'ID': ['cds-BAI77525.1'], 'Dbxref': ['NCBI_GP...",ribosomal slippage,74,6034,6036,7553,1
2,2,AB594828.1,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,slip,CDS,2,"{'ID': ['cds-BAJ72195.1'], 'Parent': ['gene-OR...",ribosomal slippage,176,1621,1621,3432,-1
3,3,AF022214.2,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,slip,CDS,2,"{'ID': ['cds-AAC18466.2'], 'Parent': ['gene-PB...",ribosomal slippage,15695,16075,16075,16518,-1
4,4,AF033808.1,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,slip,CDS,2,"{'ID': ['cds-AAC82561.1'], 'Dbxref': ['NCBI_GP...",ribosomal slippage,380,2482,2482,5190,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124873,124873,PZ005496.1,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,slip,CDS,2,"{'ID': ['cds-YDB37122.1'], 'Parent': ['gene-PA...",ribosomal slippage,13,582,584,772,1
124874,124874,PZ005504.1,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,slip,CDS,2,"{'ID': ['cds-YDB37134.1'], 'Parent': ['gene-PA...",ribosomal slippage,13,582,584,772,1
124875,124875,PZ005542.1,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,slip,CDS,2,"{'ID': ['cds-YDB37190.1'], 'Parent': ['gene-PA...",ribosomal slippage,13,582,584,772,1
124876,124876,U51190.1,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,slip,CDS,2,"{'ID': ['cds-AAC97542.1'], 'Dbxref': ['NCBI_GP...",ribosomal slippage,160,1448,1448,3146,-1


In [24]:
full = full.merge(gff_collapsed, how = 'left', on = 'record_id').drop(columns = ['source_file', 'search_term', 'type', 'qualifiers', 'exception', 'n_segments', 'start1', 'end1', 'start2', 'end2'])

In [25]:
full

,cluster,accession,record_id,Unnamed: 0,strand,slippage_direction,slippage_site
0,MW334171.1,GCA_038804155.1,MW334171.1,NaN,NaN,NaN,NaN
1,MW334171.1,GCA_039240725.1,OP023516.1,NaN,NaN,NaN,NaN
2,MW334171.1,GCA_046381355.1,PQ755305.1,NaN,NaN,NaN,NaN
3,MW334171.1,GCA_046380515.1,PQ756818.1,NaN,NaN,NaN,NaN
4,MW334171.1,GCA_046378115.1,PQ756310.1,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...
1326981,KP939613.1,GCA_003084715.1,KP939837.1,NaN,NaN,NaN,NaN
1326982,KP939613.1,GCA_003085195.1,KP940036.1,NaN,NaN,NaN,NaN
1326983,KP939613.1,GCA_003085355.1,KP940044.1,NaN,NaN,NaN,NaN
1326984,OP890528.1,GCA_027132475.1,OP890528.1,NaN,NaN,NaN,NaN


In [55]:
full.to_csv("../full.csv")

In [147]:
full = pd.read_csv('../full.csv')

In [10]:
full

,cluster,accession,record_id,Unnamed: 0,strand,slippage_direction
0,MW334171.1,GCA_038804155.1,MW334171.1,NaN,NaN,NaN
1,MW334171.1,GCA_039240725.1,OP023516.1,NaN,NaN,NaN
2,MW334171.1,GCA_046381355.1,PQ755305.1,NaN,NaN,NaN
3,MW334171.1,GCA_046380515.1,PQ756818.1,NaN,NaN,NaN
4,MW334171.1,GCA_046378115.1,PQ756310.1,NaN,NaN,NaN
...,...,...,...,...,...,...
1326981,KP939613.1,GCA_003084715.1,KP939837.1,NaN,NaN,NaN
1326982,KP939613.1,GCA_003085195.1,KP940036.1,NaN,NaN,NaN
1326983,KP939613.1,GCA_003085355.1,KP940044.1,NaN,NaN,NaN
1326984,OP890528.1,GCA_027132475.1,OP890528.1,NaN,NaN,NaN


In [26]:
import os
import glob

def get_genomic_path(accession):
    # Construct the directory path
    base_dir = f"../viral_data_all/ncbi_dataset/data*/{accession}"
    
    # Use glob to find all files ending in .fna in that specific folder
    search_pattern = os.path.join(base_dir, "*.fna")
    files = glob.glob(search_pattern)
    
    # Filter the list: exclude files that contain 'cds' (case-insensitive)
    # We look at the basename to ensure we aren't accidentally filtering based on folder names
    genomic_files = [f for f in files if 'cds' not in os.path.basename(f).lower()]
    
    # Return the first match if it exists, otherwise return None or an empty string
    return genomic_files[0] if genomic_files else None

In [27]:
assembly['fna_path'] = assembly['accession'].apply(get_genomic_path)

In [13]:
assembly

,accession,assembly_level,assembly_name,submitter,total_sequence_length,gc_percent,taxid,organism_name,release_date,fna_path
0,GCA_000320725.1,Contig,APLentillevirus_1.0,CNRS UMR IRD 6236,1193433,28.0,1077221,Acanthamoeba polyphaga lentillevirus,2012-07-27,../viral_data_all/ncbi_dataset/data_subset/GCA...
1,GCA_000847085.1,Complete Genome,ViralProj14573,"M. Jaeger, Dept of Virology, University of Ulm...",4491,33.5,1977403,Acholeplasma phage MV-L51,1993-04-21,../viral_data_all/ncbi_dataset/data/GCA_000847...
2,GCA_000848265.1,Complete Genome,ViralProj14654,"NIH, NLM",5894,55.0,11788,Abelson murine leukemia virus,1998-01-22,../viral_data_all/ncbi_dataset/data_subset/GCA...
3,GCA_000850225.1,Complete Genome,ViralProj14710,"BMC, University of Latvia",4268,44.0,154784,Acinetobacter phage AP205,2003-01-30,../viral_data_all/ncbi_dataset/data/GCA_000850...
4,GCA_000858405.1,Complete Genome,ViralProj15222,NaN,5234,40.0,185639,Acheta domestica densovirus,2002-02-04,../viral_data_all/ncbi_dataset/data/GCA_000858...
...,...,...,...,...,...,...,...,...,...,...
272510,GCF_000854525.1,Complete Genome,ViralProj14955,"Institut fuer Pflanzenvirologie, Mikrobiologie...",6624,53.5,253701,Zygocactus virus X,2004-07-21,../viral_data_all/ncbi_dataset/data_subset/GCF...
272511,GCF_000861645.1,Complete Genome,ViralProj15390,"Department of Plant Pathology, National Chung ...",9591,43.0,12232,Zucchini yellow mosaic virus,2001-11-23,../viral_data_all/ncbi_dataset/data_subset/GCF...
272512,GCF_004787635.1,Complete Genome,ASM478763v1,"Department of Microbiology & Immunobiology, Ha...",3160,37.0,114871,Zygosaccharomyces bailii virus Z,2023-05-04,../viral_data_all/ncbi_dataset/data_subset/GCF...
272513,GCF_023124225.1,Complete Genome,ASM2312422v1,"Carrington Lab, Donald Danforth Plant Science ...",5969,51.5,2501222,Zymoseptoria tritici fusarivirus 1,2023-05-04,../viral_data_all/ncbi_dataset/data_subset/GCF...


In [28]:
from ete3 import NCBITaxa

ncbi = NCBITaxa()

def get_taxonomic_lineage_info(taxid):
    """
    Given an NCBI TaxID, return two dictionaries:
      1) rank_to_taxid: { rank -> taxid }
      2) rank_to_name: { rank -> taxonomic name }
    """

    # Step 1: get lineage TaxIDs
    lineage_taxids = ncbi.get_lineage(taxid)

    # Step 2: get rank information
    taxid_to_rank = ncbi.get_rank(lineage_taxids)   # {taxid: rank}

    # Step 3: get names for all lineage taxids
    taxid_to_name = ncbi.get_taxid_translator(lineage_taxids)  # {taxid: name}

    # Step 4: build rank→taxid and rank→name mappings
    rank_to_taxid = {}
    rank_to_name = {}

    for tid in lineage_taxids:
        rank = taxid_to_rank.get(tid, None)
        if rank and rank != "no rank":
            rank_to_taxid[rank] = tid
            rank_to_name[rank] = taxid_to_name[tid]

    return rank_to_taxid, rank_to_name

def get_species_and_genus(taxid):
    """Extract species and genus names and IDs for a given taxid."""
    try:
        rank_to_taxid, rank_to_name = get_taxonomic_lineage_info(taxid)
        species = rank_to_name.get("species", None)
        genus = rank_to_name.get("genus", None)
        species_id = rank_to_taxid.get("species", None)
        genus_id = rank_to_taxid.get("genus", None)
        return species, genus, species_id, genus_id
    except Exception:
        return None, None, None, None

In [29]:
assembly[["species", "genus", "species_id", "genus_id"]] = assembly["taxid"].apply(
    lambda x: pd.Series(get_species_and_genus(x))
)

In [30]:
assembly

,accession,assembly_level,assembly_name,submitter,total_sequence_length,gc_percent,taxid,organism_name,release_date,fna_path,species,genus,species_id,genus_id
0,GCA_000320725.1,Contig,APLentillevirus_1.0,CNRS UMR IRD 6236,1193433,28.0,1077221,Acanthamoeba polyphaga lentillevirus,2012-07-27,../viral_data_all/ncbi_dataset/data_subset/GCA...,Acanthamoeba polyphaga lentillevirus,Mimivirus,1077221,315393.0
1,GCA_000847085.1,Complete Genome,ViralProj14573,"M. Jaeger, Dept of Virology, University of Ulm...",4491,33.5,1977403,Acholeplasma phage MV-L51,1993-04-21,../viral_data_all/ncbi_dataset/data/GCA_000847...,Plectrovirus L51,Plectrovirus,2170098,10875.0
2,GCA_000848265.1,Complete Genome,ViralProj14654,"NIH, NLM",5894,55.0,11788,Abelson murine leukemia virus,1998-01-22,../viral_data_all/ncbi_dataset/data_subset/GCA...,Abelson murine leukemia virus,Gammaretrovirus,11788,153135.0
3,GCA_000850225.1,Complete Genome,ViralProj14710,"BMC, University of Latvia",4268,44.0,154784,Acinetobacter phage AP205,2003-01-30,../viral_data_all/ncbi_dataset/data/GCA_000850...,Apeevirus quebecense,Apeevirus,2843696,2842379.0
4,GCA_000858405.1,Complete Genome,ViralProj15222,NaN,5234,40.0,185639,Acheta domestica densovirus,2002-02-04,../viral_data_all/ncbi_dataset/data/GCA_000858...,Scindoambidensovirus orthopteran1,Scindoambidensovirus,3052744,2733229.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
272510,GCF_000854525.1,Complete Genome,ViralProj14955,"Institut fuer Pflanzenvirologie, Mikrobiologie...",6624,53.5,253701,Zygocactus virus X,2004-07-21,../viral_data_all/ncbi_dataset/data_subset/GCF...,Potexvirus eczygocacti,Potexvirus,3433841,12176.0
272511,GCF_000861645.1,Complete Genome,ViralProj15390,"Department of Plant Pathology, National Chung ...",9591,43.0,12232,Zucchini yellow mosaic virus,2001-11-23,../viral_data_all/ncbi_dataset/data_subset/GCF...,Potyvirus cucurbitaflavitesselati,Potyvirus,3240514,12195.0
272512,GCF_004787635.1,Complete Genome,ASM478763v1,"Department of Microbiology & Immunobiology, Ha...",3160,37.0,114871,Zygosaccharomyces bailii virus Z,2023-05-04,../viral_data_all/ncbi_dataset/data_subset/GCF...,Zybavirus bailii,Zybavirus,3048462,2560258.0
272513,GCF_023124225.1,Complete Genome,ASM2312422v1,"Carrington Lab, Donald Danforth Plant Science ...",5969,51.5,2501222,Zymoseptoria tritici fusarivirus 1,2023-05-04,../viral_data_all/ncbi_dataset/data_subset/GCF...,Alphafusarivirus zymoseptoriae,Alphafusarivirus,2955279,2946790.0


In [31]:
full = full.merge(assembly, how = 'left', on = 'accession').drop(columns = ['assembly_level', 'assembly_name', 'submitter', 'organism_name', 'release_date'])

In [32]:
full

,cluster,accession,record_id,Unnamed: 0,strand,slippage_direction,slippage_site,total_sequence_length,gc_percent,taxid,fna_path,species,genus,species_id,genus_id
0,MW334171.1,GCA_038804155.1,MW334171.1,NaN,NaN,NaN,NaN,13632,44.0,11320,../viral_data_all/ncbi_dataset/data_subset/GCA...,Alphainfluenzavirus influenzae,Alphainfluenzavirus,2955291,197911.0
1,MW334171.1,GCA_039240725.1,OP023516.1,NaN,NaN,NaN,NaN,13318,45.0,11320,../viral_data_all/ncbi_dataset/data_subset/GCA...,Alphainfluenzavirus influenzae,Alphainfluenzavirus,2955291,197911.0
2,MW334171.1,GCA_046381355.1,PQ755305.1,NaN,NaN,NaN,NaN,13608,44.5,11320,../viral_data_all/ncbi_dataset/data_subset/GCA...,Alphainfluenzavirus influenzae,Alphainfluenzavirus,2955291,197911.0
3,MW334171.1,GCA_046380515.1,PQ756818.1,NaN,NaN,NaN,NaN,13600,45.0,11320,../viral_data_all/ncbi_dataset/data_subset/GCA...,Alphainfluenzavirus influenzae,Alphainfluenzavirus,2955291,197911.0
4,MW334171.1,GCA_046378115.1,PQ756310.1,NaN,NaN,NaN,NaN,13632,45.0,11320,../viral_data_all/ncbi_dataset/data_subset/GCA...,Alphainfluenzavirus influenzae,Alphainfluenzavirus,2955291,197911.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1326981,KP939613.1,GCA_003084715.1,KP939837.1,NaN,NaN,NaN,NaN,19499,42.5,86060,../viral_data_all/ncbi_dataset/data_subset/GCA...,Orbivirus alphaequi,Orbivirus,40050,10892.0
1326982,KP939613.1,GCA_003085195.1,KP940036.1,NaN,NaN,NaN,NaN,19515,42.5,86062,../viral_data_all/ncbi_dataset/data_subset/GCA...,Orbivirus alphaequi,Orbivirus,40050,10892.0
1326983,KP939613.1,GCA_003085355.1,KP940044.1,NaN,NaN,NaN,NaN,19515,42.5,86062,../viral_data_all/ncbi_dataset/data_subset/GCA...,Orbivirus alphaequi,Orbivirus,40050,10892.0
1326984,OP890528.1,GCA_027132475.1,OP890528.1,NaN,NaN,NaN,NaN,195598,33.0,10244,../viral_data_all/ncbi_dataset/data_subset/GCA...,Orthopoxvirus monkeypox,Orthopoxvirus,3431483,10242.0


In [33]:
model_data_full = full[['accession', 'record_id', 'cluster', 
'slippage_site', 'slippage_direction',
'strand', 'fna_path', 'species_id', 'genus_id', 'species', 'genus']]

In [34]:
model_data_full.columns = ['accession_id', 'record_id', 'cluster_id', 
'prf_position', 'prf_direction',
'strand', 'fna_path', 'species_taxid', 'genus_taxid', 'species_name', 'genus_name']

In [35]:
model_data_full

,accession_id,record_id,cluster_id,prf_position,prf_direction,strand,fna_path,species_taxid,genus_taxid,species_name,genus_name
0,GCA_038804155.1,MW334171.1,MW334171.1,NaN,NaN,NaN,../viral_data_all/ncbi_dataset/data_subset/GCA...,2955291,197911.0,Alphainfluenzavirus influenzae,Alphainfluenzavirus
1,GCA_039240725.1,OP023516.1,MW334171.1,NaN,NaN,NaN,../viral_data_all/ncbi_dataset/data_subset/GCA...,2955291,197911.0,Alphainfluenzavirus influenzae,Alphainfluenzavirus
2,GCA_046381355.1,PQ755305.1,MW334171.1,NaN,NaN,NaN,../viral_data_all/ncbi_dataset/data_subset/GCA...,2955291,197911.0,Alphainfluenzavirus influenzae,Alphainfluenzavirus
3,GCA_046380515.1,PQ756818.1,MW334171.1,NaN,NaN,NaN,../viral_data_all/ncbi_dataset/data_subset/GCA...,2955291,197911.0,Alphainfluenzavirus influenzae,Alphainfluenzavirus
4,GCA_046378115.1,PQ756310.1,MW334171.1,NaN,NaN,NaN,../viral_data_all/ncbi_dataset/data_subset/GCA...,2955291,197911.0,Alphainfluenzavirus influenzae,Alphainfluenzavirus
...,...,...,...,...,...,...,...,...,...,...,...
1326981,GCA_003084715.1,KP939837.1,KP939613.1,NaN,NaN,NaN,../viral_data_all/ncbi_dataset/data_subset/GCA...,40050,10892.0,Orbivirus alphaequi,Orbivirus
1326982,GCA_003085195.1,KP940036.1,KP939613.1,NaN,NaN,NaN,../viral_data_all/ncbi_dataset/data_subset/GCA...,40050,10892.0,Orbivirus alphaequi,Orbivirus
1326983,GCA_003085355.1,KP940044.1,KP939613.1,NaN,NaN,NaN,../viral_data_all/ncbi_dataset/data_subset/GCA...,40050,10892.0,Orbivirus alphaequi,Orbivirus
1326984,GCA_027132475.1,OP890528.1,OP890528.1,NaN,NaN,NaN,../viral_data_all/ncbi_dataset/data_subset/GCA...,3431483,10242.0,Orthopoxvirus monkeypox,Orthopoxvirus


In [36]:
model_data_full.prf_direction.value_counts()

prf_direction
 1.0       120343
-1.0         4364
-2.0          120
 2.0           16
-4.0            9
 3.0            8
 5.0            2
 395.0          2
 71.0           2
 496.0          2
-8.0            2
 177.0          1
 61.0           1
-5.0            1
 688.0          1
 104.0          1
 5196.0         1
 12.0           1
-10.0           1
Name: count, dtype: int64

In [37]:
model_data_full.to_csv("../model_data_full2.csv")

In [40]:
string = pd.read_csv("../model_data_full_string2.csv")

In [41]:
string

,Unnamed: 0,accession_id,record_id,cluster_id,prf_position,prf_direction,strand,fna_path,species_taxid,genus_taxid,species_name,genus_name,sequence
0,0,GCA_038804155.1,MW334171.1,MW334171.1,NaN,NaN,NaN,../viral_data_all/ncbi_dataset/data_subset/GCA...,2955291,197911.0,Alphainfluenzavirus influenzae,Alphainfluenzavirus,AGCAAAAGCAGGTCAATTATATTCAATATGGAGAGAATAAAAGAAC...
1,1,GCA_039240725.1,OP023516.1,MW334171.1,NaN,NaN,NaN,../viral_data_all/ncbi_dataset/data_subset/GCA...,2955291,197911.0,Alphainfluenzavirus influenzae,Alphainfluenzavirus,AATATGGAGAGAATAAAAGAGCTAAGAGATTTGATGTCGCAGTCTC...
2,2,GCA_046381355.1,PQ755305.1,MW334171.1,NaN,NaN,NaN,../viral_data_all/ncbi_dataset/data_subset/GCA...,2955291,197911.0,Alphainfluenzavirus influenzae,Alphainfluenzavirus,AGCAAAAGCAGGTCAAATATATTCAATATGGAGAGAATAAAAGAGC...
3,3,GCA_046380515.1,PQ756818.1,MW334171.1,NaN,NaN,NaN,../viral_data_all/ncbi_dataset/data_subset/GCA...,2955291,197911.0,Alphainfluenzavirus influenzae,Alphainfluenzavirus,AGCAAAAGCAGGTCAAATATATTCAATATGGAGAGAATAAAAGAGC...
4,4,GCA_046378115.1,PQ756310.1,MW334171.1,NaN,NaN,NaN,../viral_data_all/ncbi_dataset/data_subset/GCA...,2955291,197911.0,Alphainfluenzavirus influenzae,Alphainfluenzavirus,AGCAAAAGCAGGTCAAATATATTCAATATGGAGAGAATAAAAGAGC...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1326981,1326981,GCA_003084715.1,KP939837.1,KP939613.1,NaN,NaN,NaN,../viral_data_all/ncbi_dataset/data_subset/GCA...,40050,10892.0,Orbivirus alphaequi,Orbivirus,GTTTATTTAGGATGGAACCTTACGCAATATTGTATGTTACGCAGGA...
1326982,1326982,GCA_003085195.1,KP940036.1,KP939613.1,NaN,NaN,NaN,../viral_data_all/ncbi_dataset/data_subset/GCA...,40050,10892.0,Orbivirus alphaequi,Orbivirus,GTTTATTTAGGATGGAACCTTACGCAATATTGTATGTTACGCAGGA...
1326983,1326983,GCA_003085355.1,KP940044.1,KP939613.1,NaN,NaN,NaN,../viral_data_all/ncbi_dataset/data_subset/GCA...,40050,10892.0,Orbivirus alphaequi,Orbivirus,GTTTATTTAGGATGGAACCTTACGCAATATTGTATGTTACGCAGGA...
1326984,1326984,GCA_027132475.1,OP890528.1,OP890528.1,NaN,NaN,NaN,../viral_data_all/ncbi_dataset/data_subset/GCA...,3431483,10242.0,Orthopoxvirus monkeypox,Orthopoxvirus,GTGGTGCGGTGTCTGAATCGTTCGATTAACCCAACTCATCCATTTT...


In [113]:
string.rename(columns = {'strand_record': 'strand'}, inplace = True)

In [115]:
string['strand'] = 1

In [117]:
string.to_csv("../model_data_full_string_final.csv")

In [54]:
string[string.prf_direction == 0]

,Unnamed: 0,accession_id,record_id,cluster_id,prf_position,prf_direction,start1,end1,start2,end2,strand,fna_path,species_taxid,genus_taxid,species_name,genus_name,sequence
694,694,GCA_046887915.1,PP895394.1,PP895394.1,570.0,0.0,0.0,570.0,571.0,760.0,1.0,../viral_data_all/ncbi_dataset/data_subset/GCA...,2955291,197911.0,Alphainfluenzavirus influenzae,Alphainfluenzavirus,ATGGAAGACTTTGTGCGACAATGCTTCAATCCAATGGTCGTCGAGC...
696,696,GCF_002890315.1,NC_036580.1,NC_036580.1,982.0,0.0,148.0,982.0,983.0,3323.0,1.0,../viral_data_all/ncbi_dataset/data_subset/GCF...,3047357,1511861.0,Amalgavirus allii,Amalgavirus,GGGTATAAATAAAATTCTTATTCGTAGGACCTCATTATACTTGCCT...
697,697,GCA_002890315.1,BK010347.1,NC_036580.1,982.0,0.0,148.0,982.0,983.0,3323.0,1.0,../viral_data_all/ncbi_dataset/data_subset/GCA...,3047357,1511861.0,Amalgavirus allii,Amalgavirus,GGGTATAAATAAAATTCTTATTCGTAGGACCTCATTATACTTGCCT...
4341,4341,GCA_039168845.1,MW159331.1,MW159331.1,594.0,0.0,24.0,594.0,595.0,784.0,1.0,../viral_data_all/ncbi_dataset/data_subset/GCA...,2955291,197911.0,Alphainfluenzavirus influenzae,Alphainfluenzavirus,AGCAAAAGCAGGTACTGATCCAAAATGGAAGACTTTGTGCGACAGT...
4342,4342,GCA_039055315.1,MZ543300.1,MW159331.1,582.0,0.0,12.0,582.0,583.0,772.0,1.0,../viral_data_all/ncbi_dataset/data_subset/GCA...,2955291,197911.0,Alphainfluenzavirus influenzae,Alphainfluenzavirus,TACTGATCCGAAATGGAAGACTTTGTGCGTCAGTGCTTCAATCCAA...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1326461,1326461,GCA_038642845.1,KX595257.1,KX595257.1,570.0,0.0,0.0,570.0,571.0,760.0,1.0,../viral_data_all/ncbi_dataset/data_subset/GCA...,2955291,197911.0,Alphainfluenzavirus influenzae,Alphainfluenzavirus,ATGGAAGACCTTGTGCGACAATGCTTCAATCCAATGATCGTCGAGC...
1326462,1326462,GCA_038643595.1,KX595266.1,KX595257.1,570.0,0.0,0.0,570.0,571.0,760.0,1.0,../viral_data_all/ncbi_dataset/data_subset/GCA...,2955291,197911.0,Alphainfluenzavirus influenzae,Alphainfluenzavirus,ATGGAAGACCTTGTGCGACAATGCTTCAATCCAATGATCGTCGAGC...
1326906,1326906,GCA_049040145.1,PV388414.1,PV388414.1,537.0,0.0,0.0,537.0,538.0,727.0,1.0,../viral_data_all/ncbi_dataset/data_subset/GCA...,2955291,197911.0,Alphainfluenzavirus influenzae,Alphainfluenzavirus,ATGATTGTCGAGCTTGCGGAAAAGGCAATGAARGARTATGGGGAAG...
1326940,1326940,GCA_000954375.1,KM101124.1,KM101124.1,8887.0,0.0,8389.0,8887.0,8888.0,9326.0,1.0,../viral_data_all/ncbi_dataset/data_subset/GCA...,3432580,2843461.0,Squirtyvirus squirty,Squirtyvirus,AGGCGAGTGTGTGTGTGACCCAGGGGTAGGGAGGTTGTAGTGCCGT...


In [125]:
string[string.cluster_id == string.record_id].drop_duplicates(subset = 'cluster_id')

,Unnamed: 0,accession_id,record_id,cluster_id,prf_position,strand,fna_path,species_taxid,genus_taxid,species_name,genus_name,sequence
0,0,GCA_038804155.1,MW334171.1,MW334171.1,NaN,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,2955291,197911.0,Alphainfluenzavirus influenzae,Alphainfluenzavirus,AGCAAAAGCAGGTCAATTATATTCAATATGGAGAGAATAAAAGAAC...
522,522,GCA_003083835.1,KT030548.1,KT030548.1,NaN,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,40050,10892.0,Orbivirus alphaequi,Orbivirus,GTTTAAAAATCCGTTCGTCATCATGGCAGAGGTCAGAAAGCAACAA...
692,692,GCA_055130685.1,PX974797.1,PX974797.1,NaN,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,2955291,197911.0,Alphainfluenzavirus influenzae,Alphainfluenzavirus,ATGGATGTCAATCCGACTCTATTGTTCCTAAAAGTTCCAGCGCAAA...
693,693,GCA_025739875.1,OP414386.1,OP414386.1,NaN,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,3431483,10242.0,Orthopoxvirus monkeypox,Orthopoxvirus,TNNTACAGATCATTTATATTCCAAAAATATTAACTATATACGTTTA...
694,694,GCA_046887915.1,PP895394.1,PP895394.1,570.0,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,2955291,197911.0,Alphainfluenzavirus influenzae,Alphainfluenzavirus,ATGGAAGACTTTGTGCGACAATGCTTCAATCCAATGGTCGTCGAGC...
...,...,...,...,...,...,...,...,...,...,...,...,...
1326942,1326942,GCA_003442215.1,MH669001.1,MH669001.1,9072.0,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,2955631,1623281.0,Cheoctovirus eleanorgeorge,Cheoctovirus,TGCAGATTTTGGTCTGTACGGAACCGGGGGGTTTCGCGGCATCCCC...
1326944,1326944,GCA_020496375.1,MZ375253.1,MZ375253.1,NaN,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,1985738,110456.0,Teseptimavirus T7,Teseptimavirus,TCTCACAGTGTACGGACCTAAAGTTCCCCCATAGGGGGTACCTAAA...
1326946,1326946,GCA_003084335.1,KP939613.1,KP939613.1,NaN,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,40050,10892.0,Orbivirus alphaequi,Orbivirus,GTTTATTTAGGATGGAACCTTACGCAATATTGTATGTTACGCAGGA...
1326984,1326984,GCA_027132475.1,OP890528.1,OP890528.1,NaN,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,3431483,10242.0,Orthopoxvirus monkeypox,Orthopoxvirus,GTGGTGCGGTGTCTGAATCGTTCGATTAACCCAACTCATCCATTTT...


In [126]:
string[string.cluster_id != string.record_id]

,Unnamed: 0,accession_id,record_id,cluster_id,prf_position,strand,fna_path,species_taxid,genus_taxid,species_name,genus_name,sequence
1,1,GCA_039240725.1,OP023516.1,MW334171.1,NaN,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,2955291,197911.0,Alphainfluenzavirus influenzae,Alphainfluenzavirus,AATATGGAGAGAATAAAAGAGCTAAGAGATTTGATGTCGCAGTCTC...
2,2,GCA_046381355.1,PQ755305.1,MW334171.1,NaN,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,2955291,197911.0,Alphainfluenzavirus influenzae,Alphainfluenzavirus,AGCAAAAGCAGGTCAAATATATTCAATATGGAGAGAATAAAAGAGC...
3,3,GCA_046380515.1,PQ756818.1,MW334171.1,NaN,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,2955291,197911.0,Alphainfluenzavirus influenzae,Alphainfluenzavirus,AGCAAAAGCAGGTCAAATATATTCAATATGGAGAGAATAAAAGAGC...
4,4,GCA_046378115.1,PQ756310.1,MW334171.1,NaN,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,2955291,197911.0,Alphainfluenzavirus influenzae,Alphainfluenzavirus,AGCAAAAGCAGGTCAAATATATTCAATATGGAGAGAATAAAAGAGC...
5,5,GCA_038953265.1,MN874429.1,MW334171.1,NaN,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,2955291,197911.0,Alphainfluenzavirus influenzae,Alphainfluenzavirus,TAAGAGATTTGATGTCGCAGTCTCGCACTCGCGAGATACTGACAAA...
...,...,...,...,...,...,...,...,...,...,...,...,...
1326979,1326979,GCA_003085455.1,KP940157.1,KP939613.1,NaN,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,40050,10892.0,Orbivirus alphaequi,Orbivirus,GTTTATTTAGGATGGAACCTTACGCAATACTGTATGTTACGCAGGA...
1326980,1326980,GCA_003084295.1,KP939611.1,KP939613.1,NaN,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,40050,10892.0,Orbivirus alphaequi,Orbivirus,GTTTATTTAGGATGGAACCTTACGCAATATTGTATGTTACGCAGGA...
1326981,1326981,GCA_003084715.1,KP939837.1,KP939613.1,NaN,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,40050,10892.0,Orbivirus alphaequi,Orbivirus,GTTTATTTAGGATGGAACCTTACGCAATATTGTATGTTACGCAGGA...
1326982,1326982,GCA_003085195.1,KP940036.1,KP939613.1,NaN,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,40050,10892.0,Orbivirus alphaequi,Orbivirus,GTTTATTTAGGATGGAACCTTACGCAATATTGTATGTTACGCAGGA...


count
1        33507
2        10952
3         1439
4          879
5          440
         ...  
47904        1
48180        1
48286        1
48288        1
48298        1
Name: count, Length: 461, dtype: int64

In [142]:
string.strand_record.value_counts()

strand_record
 1.0    124822
-1.0        56
Name: count, dtype: int64

In [144]:
temp = string[string.strand_record == -1]

In [145]:
temp.to_csv("../record_negative_strand.csv")

In [149]:
full

,Unnamed: 0,cluster,accession,record_id,strand,slippage_site,slippage_direction
0,0,MW334171.1,GCA_038804155.1,MW334171.1,NaN,NaN,NaN
1,1,MW334171.1,GCA_039240725.1,OP023516.1,NaN,NaN,NaN
2,2,MW334171.1,GCA_046381355.1,PQ755305.1,NaN,NaN,NaN
3,3,MW334171.1,GCA_046380515.1,PQ756818.1,NaN,NaN,NaN
4,4,MW334171.1,GCA_046378115.1,PQ756310.1,NaN,NaN,NaN
...,...,...,...,...,...,...,...
1326981,1326981,KP939613.1,GCA_003084715.1,KP939837.1,NaN,NaN,NaN
1326982,1326982,KP939613.1,GCA_003085195.1,KP940036.1,NaN,NaN,NaN
1326983,1326983,KP939613.1,GCA_003085355.1,KP940044.1,NaN,NaN,NaN
1326984,1326984,OP890528.1,GCA_027132475.1,OP890528.1,NaN,NaN,NaN


In [150]:
string

,Unnamed: 0,accession_id,record_id,cluster_id,prf_position,strand_record,fna_path,species_taxid,genus_taxid,species_name,genus_name,sequence
0,0,GCA_038804155.1,MW334171.1,MW334171.1,NaN,NaN,../viral_data_all/ncbi_dataset/data_subset/GCA...,2955291,197911.0,Alphainfluenzavirus influenzae,Alphainfluenzavirus,AGCAAAAGCAGGTCAATTATATTCAATATGGAGAGAATAAAAGAAC...
1,1,GCA_039240725.1,OP023516.1,MW334171.1,NaN,NaN,../viral_data_all/ncbi_dataset/data_subset/GCA...,2955291,197911.0,Alphainfluenzavirus influenzae,Alphainfluenzavirus,AATATGGAGAGAATAAAAGAGCTAAGAGATTTGATGTCGCAGTCTC...
2,2,GCA_046381355.1,PQ755305.1,MW334171.1,NaN,NaN,../viral_data_all/ncbi_dataset/data_subset/GCA...,2955291,197911.0,Alphainfluenzavirus influenzae,Alphainfluenzavirus,AGCAAAAGCAGGTCAAATATATTCAATATGGAGAGAATAAAAGAGC...
3,3,GCA_046380515.1,PQ756818.1,MW334171.1,NaN,NaN,../viral_data_all/ncbi_dataset/data_subset/GCA...,2955291,197911.0,Alphainfluenzavirus influenzae,Alphainfluenzavirus,AGCAAAAGCAGGTCAAATATATTCAATATGGAGAGAATAAAAGAGC...
4,4,GCA_046378115.1,PQ756310.1,MW334171.1,NaN,NaN,../viral_data_all/ncbi_dataset/data_subset/GCA...,2955291,197911.0,Alphainfluenzavirus influenzae,Alphainfluenzavirus,AGCAAAAGCAGGTCAAATATATTCAATATGGAGAGAATAAAAGAGC...
...,...,...,...,...,...,...,...,...,...,...,...,...
1326981,1326981,GCA_003084715.1,KP939837.1,KP939613.1,NaN,NaN,../viral_data_all/ncbi_dataset/data_subset/GCA...,40050,10892.0,Orbivirus alphaequi,Orbivirus,GTTTATTTAGGATGGAACCTTACGCAATATTGTATGTTACGCAGGA...
1326982,1326982,GCA_003085195.1,KP940036.1,KP939613.1,NaN,NaN,../viral_data_all/ncbi_dataset/data_subset/GCA...,40050,10892.0,Orbivirus alphaequi,Orbivirus,GTTTATTTAGGATGGAACCTTACGCAATATTGTATGTTACGCAGGA...
1326983,1326983,GCA_003085355.1,KP940044.1,KP939613.1,NaN,NaN,../viral_data_all/ncbi_dataset/data_subset/GCA...,40050,10892.0,Orbivirus alphaequi,Orbivirus,GTTTATTTAGGATGGAACCTTACGCAATATTGTATGTTACGCAGGA...
1326984,1326984,GCA_027132475.1,OP890528.1,OP890528.1,NaN,NaN,../viral_data_all/ncbi_dataset/data_subset/GCA...,3431483,10242.0,Orthopoxvirus monkeypox,Orthopoxvirus,GTGGTGCGGTGTCTGAATCGTTCGATTAACCCAACTCATCCATTTT...


In [151]:
string[string.duplicated(subset=["record_id", 'cluster_id'])]

,Unnamed: 0,accession_id,record_id,cluster_id,prf_position,strand_record,fna_path,species_taxid,genus_taxid,species_name,genus_name,sequence
16539,16539,GCA_003814185.1,MK072466.1,MK072466.1,NaN,NaN,../viral_data_all/ncbi_dataset/data_subset/GCA...,2487771,NaN,Satyrvirus sp.,NaN,TGCGCGAATTTCATACGATAAGTTCATAATTATATGTAGTTCTCGT...
16540,16540,GCA_003810345.1,MK072466.1,MK072466.1,NaN,NaN,../viral_data_all/ncbi_dataset/data_subset/GCA...,2487771,NaN,Satyrvirus sp.,NaN,TGCGCGAATTTCATACGATAAGTTCATAATTATATGTAGTTCTCGT...
16541,16541,GCA_003814185.1,MK072466.1,MK072466.1,NaN,NaN,../viral_data_all/ncbi_dataset/data_subset/GCA...,2487771,NaN,Satyrvirus sp.,NaN,TGCGCGAATTTCATACGATAAGTTCATAATTATATGTAGTTCTCGT...
19891,19891,GCA_004117175.3,LC320125.1,NC_040732.1,NaN,NaN,../viral_data_all/ncbi_dataset/data_subset/GCA...,2956525,35323.0,Thogotovirus ozense,Thogotovirus,AGCAAAAACAGGCAGATATTCCAACAGAATGGACAAATTTAGACCC...
19892,19892,GCA_004117175.1,LC320125.1,NC_040732.1,NaN,NaN,../viral_data_all/ncbi_dataset/data_subset/GCA...,2956525,35323.0,Thogotovirus ozense,Thogotovirus,AGCAAAAACAGGCAGATATTCCAACAGAATGGACAAATTTAGACCC...
...,...,...,...,...,...,...,...,...,...,...,...,...
1323051,1323051,GCA_003810385.1,MK072468.1,MK072468.1,NaN,NaN,../viral_data_all/ncbi_dataset/data_subset/GCA...,2487771,NaN,Satyrvirus sp.,NaN,ACGGAGGGCCGCAAATGTTTCATACAAGAACCGGTAGCGGCCCTTC...
1323052,1323052,GCA_003814185.1,MK072468.1,MK072468.1,NaN,NaN,../viral_data_all/ncbi_dataset/data_subset/GCA...,2487771,NaN,Satyrvirus sp.,NaN,ACGGAGGGCCGCAAATGTTTCATACAAGAACCGGTAGCGGCCCTTC...
1326883,1326883,GCA_000854765.2,L25784.1,NC_077669.1,NaN,NaN,../viral_data_all/ncbi_dataset/data_subset/GCA...,3052499,1980442.0,Orthohantavirus sinnombreense,Orthohantavirus,TAGTAGTAGACTCCTTGAGAAGCTACTACGACTAAAGCTGGAATGA...
1326884,1326884,GCA_000854765.1,L25784.1,NC_077669.1,NaN,NaN,../viral_data_all/ncbi_dataset/data_subset/GCA...,3052499,1980442.0,Orthohantavirus sinnombreense,Orthohantavirus,TAGTAGTAGACTCCTTGAGAAGCTACTACGACTAAAGCTGGAATGA...


In [152]:
string[string.prf_position.notna()]

,Unnamed: 0,accession_id,record_id,cluster_id,prf_position,strand_record,fna_path,species_taxid,genus_taxid,species_name,genus_name,sequence
694,694,GCA_046887915.1,PP895394.1,PP895394.1,570.0,1.0,../viral_data_all/ncbi_dataset/data_subset/GCA...,2955291,197911.0,Alphainfluenzavirus influenzae,Alphainfluenzavirus,ATGGAAGACTTTGTGCGACAATGCTTCAATCCAATGGTCGTCGAGC...
696,696,GCF_002890315.1,NC_036580.1,NC_036580.1,982.0,1.0,../viral_data_all/ncbi_dataset/data_subset/GCF...,3047357,1511861.0,Amalgavirus allii,Amalgavirus,GGGTATAAATAAAATTCTTATTCGTAGGACCTCATTATACTTGCCT...
697,697,GCA_002890315.1,BK010347.1,NC_036580.1,982.0,1.0,../viral_data_all/ncbi_dataset/data_subset/GCA...,3047357,1511861.0,Amalgavirus allii,Amalgavirus,GGGTATAAATAAAATTCTTATTCGTAGGACCTCATTATACTTGCCT...
1423,1423,GCA_026724305.1,OP751151.1,OP751151.1,9352.0,1.0,../viral_data_all/ncbi_dataset/data_subset/GCA...,2999016,1982076.0,Arthrobacter phage Chridison,Korravirus,GTGTCCTCTCCCTCCCCGGCATTCGACCCGGGGTCGCCCGCACACG...
4341,4341,GCA_039168845.1,MW159331.1,MW159331.1,594.0,1.0,../viral_data_all/ncbi_dataset/data_subset/GCA...,2955291,197911.0,Alphainfluenzavirus influenzae,Alphainfluenzavirus,AGCAAAAGCAGGTACTGATCCAAAATGGAAGACTTTGTGCGACAGT...
...,...,...,...,...,...,...,...,...,...,...,...,...
1326906,1326906,GCA_049040145.1,PV388414.1,PV388414.1,537.0,1.0,../viral_data_all/ncbi_dataset/data_subset/GCA...,2955291,197911.0,Alphainfluenzavirus influenzae,Alphainfluenzavirus,ATGATTGTCGAGCTTGCGGAAAAGGCAATGAARGARTATGGGGAAG...
1326940,1326940,GCA_000954375.1,KM101124.1,KM101124.1,8887.0,1.0,../viral_data_all/ncbi_dataset/data_subset/GCA...,3432580,2843461.0,Squirtyvirus squirty,Squirtyvirus,AGGCGAGTGTGTGTGTGACCCAGGGGTAGGGAGGTTGTAGTGCCGT...
1326941,1326941,GCF_000954375.1,NC_026588.1,KM101124.1,8887.0,1.0,../viral_data_all/ncbi_dataset/data_subset/GCF...,3432580,2843461.0,Squirtyvirus squirty,Squirtyvirus,AGGCGAGTGTGTGTGTGACCCAGGGGTAGGGAGGTTGTAGTGCCGT...
1326942,1326942,GCA_003442215.1,MH669001.1,MH669001.1,9072.0,1.0,../viral_data_all/ncbi_dataset/data_subset/GCA...,2955631,1623281.0,Cheoctovirus eleanorgeorge,Cheoctovirus,TGCAGATTTTGGTCTGTACGGAACCGGGGGGTTTCGCGGCATCCCC...
